In [1]:
import os
from dotenv import load_dotenv
from IPython.display import display, display_markdown
from openai import OpenAI
import pandas as pd
from rich import print as pprint

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
import openai
openai.__version__

'1.99.8'

In [ ]:
바쁘신 개발자분들을 위해서 gpt-5 에서 새롭게 생긴점에 대해서 간략히 설명드리도록 하겠습니다. 

- gpt-5 모델은 temperature 파라미터를 지원하지 않습니다.
- verbosity 옵션으로 모델 응답의 길이를 조절 할 수 있습니다.

## Temperature 

gpt5 계열 모델은 temperature 을 (아직) 지원하지 않는다. temperature 로 퀄리티를 조절 하셨던 분들은 조금 당황 스러우셨을 수 있을 것 같습니다.

In [22]:
resp = client.responses.create(
    model="gpt-5-mini",
    input="hi",
    temperature=0.
)

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}

## Verbosity

In [3]:
import tiktoken

def count_token(target, model_name):
    # enc_model_name = tiktoken.encoding_name_for_model(model_name)
    enc_model = tiktoken.encoding_for_model(model_name)
    return len(enc_model.encode(target))

In [53]:
def change_to_df(data, trial=None, display=False):
    # Create DataFrame
    output_df = pd.DataFrame(data)
    if trial:
        output_df['trial'] = trial

    if display:
        # Display nicely with centered headers
        pd.set_option('display.max_colwidth', None)
        styled_df = output_df.style.set_table_styles(
            [
                {'selector': 'th', 'props': [('text-align', 'center')]},  # Center column headers
                {'selector': 'td', 'props': [('text-align', 'left')]}     # Left-align table cells
            ]
        )

        display(styled_df)

    return output_df

def display_df(dataframe):
    pd.set_option('display.max_colwidth', None)
    styled_df = dataframe.style.set_table_styles(
        [
            {'selector': 'th', 'props': [('text-align', 'center')]},  # Center column headers
            {'selector': 'td', 'props': [('text-align', 'left')]}     # Left-align table cells
        ]
    )

    display(styled_df)

In [5]:
def generate_multiple_outputs(query, configurations, model="gpt-5-mini", display_results=True):
    """
    Generate multiple outputs with different configurations and optionally display them.
    
    Args:
        query (str): The input query/prompt
        configurations (list): List of configuration dictionaries for different outputs
        model (str): The model to use (default: "gpt-5-mini")
        display_results (bool): Whether to display the results using display_output
    
    Returns:
        tuple: (raw_responses, formatted_outputs)
        
    Example configurations:
    [
        {"name": "Low Verbosity", "text": {"verbosity": "low"}},
        {"name": "Medium Verbosity", "text": {"verbosity": "medium"}}, 
        {"name": "High Verbosity", "text": {"verbosity": "high"}},
        {"name": "With Reasoning", "reasoning": {"summary": "auto"}},
    ]
    """
    raw_responses = []
    outputs = []
    
    for config in configurations:
        # Extract configuration name
        config_name = config.pop("name", "Unknown")
        
        # Create response with the configuration
        resp = client.responses.create(
            model=model,
            input=query,
            **config
        )
        raw_responses.append(resp)
        
        # Extract reasoning summary if available
        reason_summaries = []
        for item in resp.output:
            if getattr(item, "type", None) == "reasoning" and getattr(item, "summary", None):
                for seg in item.summary:
                    if getattr(seg, "text", None):
                        reason_summaries.append(seg.text)
        reasoning_output = "\n".join(reason_summaries)
        
        # Prepare output data
        output_data = {
            "Configuration": config_name,
            "Sample Output": resp.output_text,
            "Reasoning Output": reasoning_output if reasoning_output else "N/A",
            "Output Tokens": resp.usage.output_tokens,
            "Reasoning Tokens": getattr(resp.usage.output_tokens_details, 'reasoning_tokens', 0),
            "Total Tokens": resp.usage.total_tokens
        }
        outputs.append(output_data)
    
    if display_results:
        change_to_df(outputs, display=True)
    
    return raw_responses, outputs

In [36]:
# Test query
query = "안녕하세요"

# Generate outputs
num_trial = 5

history = dict()
for trial in range(num_trial):
    # Configuration for different verbosity levels
    verbosity_configs = [
        {"name": "Low Verbosity", "text": {"verbosity": "low"}, "reasoning": {"effort": "medium", "summary": "auto"}},
        {"name": "Medium Verbosity", "text": {"verbosity": "medium"}, "reasoning": {"effort": "medium", "summary": "auto"}}, 
        {"name": "High Verbosity", "text": {"verbosity": "high"}, "reasoning": {"effort": "medium", "summary": "auto"}}
    ]
    raw_responses, outputs = generate_multiple_outputs(query, verbosity_configs, display_results=False)
    history[trial+1] = {"raw_responses": raw_responses, "outputs": outputs}

In [37]:
from datetime import datetime
import json


dfs = []
for k, v in history.items():
    output_df = change_to_df(v["outputs"], k, display=False)
    dfs.append(output_df)

# Save history to JSON file for debugging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"openai_test_history_ko_{timestamp}.json"


print(f"History saved to {filename}")
final_df = pd.concat(dfs, ignore_index=True)
final_df.to_json(filename)

History saved to openai_test_history_ko_20250812_122644.json


In [39]:
final_df["Reasoning Tokens"]

0      64
1       0
2     192
3      64
4     128
5     256
6      64
7      64
8     192
9      64
10     64
11    192
12     64
13      0
14    256
Name: Reasoning Tokens, dtype: int64

verbosity 옵션에는 3가지 'low', 'medium', 'high' 옵션을 지원합니다. 기본값은 'medium' 입니다.
 '안녕하세요' 라는 요청에 대한 예시는 옵션에 따른 출력은 다음과 같습니다.

In [59]:
# display_df(final_df)
trial = 1
selected = final_df.set_index("trial").loc[trial]
for _, row in selected.iterrows():
    pprint(f"Configuration: {row.Configuration}")
    pprint(row["Sample Output"])
    pprint("-"*50)

Configuration: Low Verbosity

안녕하세요! 무엇을 도와드릴까요?

--------------------------------------------------

Configuration: Medium Verbosity

안녕하세요! 만나서 반갑습니다. 어떻게 도와드릴까요?

--------------------------------------------------

Configuration: High Verbosity

안녕하세요! 반갑습니다 — 무엇을 도와드릴까요?

간단히 예를 들면 다음과 같이 도와드릴 수 있어요:
- 번역(한국어↔영어 등), 문장 다듬기, 이메일·보고서·자기소개서 작성
- 요약(기사·문서), 정보 검색과 설명(역사·과학·법률 기초 등)
- 코드 작성·디버깅(파이썬, 자바스크립트 등), 알고리즘 설명
- 여행 일정·식단·운동 계획 세우기
- 문제 풀이(수학·물리), 언어 학습 도움
- 이미지가 있으면 분석하거나 설명해드릴 수도 있어요(사진 업로드해 주세요)

원하시면 편한 말(반말)로 바꿔 드릴 수도 있고, 더 공손한 표현으로도 도와드릴게요. 무엇을 하실래요?

--------------------------------------------------

간헐적으로 특이한 현상이 관찰 할 수 있었는데, token usage 에 reasoning token 사용량은 0 으로 잡히는데, reasoning summary 가 존재하는 경우가 종종 관찰되었습니다.

In [61]:
display_df(final_df[final_df["Reasoning Tokens"]==0])

,Configuration,Sample Output,Reasoning Output,Output Tokens,Reasoning Tokens,Total Tokens,trial
1,Medium Verbosity,안녕하세요! 만나서 반갑습니다. 어떻게 도와드릴까요?,"I'm needing to respond to the user in Korean since they greeted me with ""안녕하세요."" I'll keep it polite and concise, perhaps saying something like, ""안녕하세요! 무엇을 도와드릴까요?"" This also opens the door to ask about their language preferences. It's essential to respond warmly to make the conversation inviting! Let’s keep things friendly and helpful!",22,0,30,1
13,Medium Verbosity,안녕하세요! 어떻게 도와드릴까요?,안녕하세요! 반갑습니다. 어떻게 도와드릴까요? 원하는 언어로 대화할 수 있어요. 무엇을 하고 싶으신가요?,16,0,24,5


### verbosity - code

코드의 경우 에도 verbosity 로 인한 차이를 볼 수 있습니다. verbosity 가 높아질 수록, 코드에 대한 주석과 설명의 양이 늘어나는거 볼 수 있습니다.

In [65]:
def ask_with_verbosity(verbosity: str, question: str, model="gpt-5-mini"):
    response = client.responses.create(
        model=model,
        input=question,
        text={
            "verbosity": verbosity
        }
    )

    # Extract assistant's text output
    output_text = ""
    for item in response.output:
        if hasattr(item, "content") and item.content:
            for content in item.content:
                if hasattr(content, "text"):
                    output_text += content.text

    # Token usage details
    usage = response.usage

    print("--------------------------------")
    print(f"Verbosity: {verbosity}")
    print("Output:")
    # print(output_text)
    print("Tokens => input: {} | output: {}".format(
        usage.input_tokens, usage.output_tokens
    ))

    return output_text

In [66]:
prompt = "Output a Python program that sorts an array of 1000000 random numbers"

low_result = ask_with_verbosity("low", prompt)
medium_result = ask_with_verbosity("medium", prompt)
high_result = ask_with_verbosity("high", prompt)


--------------------------------
Verbosity: low
Output:
Tokens => input: 21 | output: 779
--------------------------------
Verbosity: medium
Output:
Tokens => input: 21 | output: 807
--------------------------------
Verbosity: high
Output:
Tokens => input: 21 | output: 2153


In [156]:
pprint(low_result)

#!/usr/bin/env python3
import random
import time

def main():
    n = 1_000_000
    random.seed(0)  # remove or change seed for different data
    arr = 

    t0 = time.perf_counter()
    arr.sort()
    t1 = time.perf_counter()

    print(f"Sorted {n} numbers in {t1 - t0:.4f} seconds")
    print("First 10:", arr[:10])
    print("Last 10:", arr[-10:])

if __name__ == "__main__":
    main()

In [157]:
pprint(medium_result)

#!/usr/bin/env python3
import random
import time

def main():
    n = 1_000_000  # number of random numbers
    random.seed(0)  # optional: reproducible results

    # Generate the list of random floats
    arr = 

    # Sort in place and time it
    t0 = time.perf_counter()
    arr.sort()
    t1 = time.perf_counter()

    print(f"Sorted {n} numbers in {t1 - t0:.4f} seconds.")
    # sanity checks
    print("First 10:", arr[:10])
    print("Last 10: ", arr[-10:])

if __name__ == "__main__":
    main()

In [158]:
pprint(high_result)

Here's a self-contained Python program that generates 1,000,000 random numbers and sorts them. It times the 
generation and sorting steps, optionally uses NumPy (faster / more memory-efficient) if you pass --numpy, and 
verifies the result is sorted.

Save as sort_random.py and run with python sort_random.py

Code:

```python
#!/usr/bin/env python3
"""
Generate and sort N random numbers, timing the operations.

Usage:
    python sort_random.py            # default: 1_000_000 floats, pure Python
    python sort_random.py --n 10000  # change count
    python sort_random.py --numpy    # use numpy if available (faster)
    python sort_random.py --seed 123 # reproducible (Python RNG)
"""

import time
import random
import argparse
import sys

def parse_args():
    p = argparse.ArgumentParser(description="Generate and sort random numbers.")
    p.add_argument("--n", type=int, default=1_000_000, help="How many random numbers to generate (default: 
1_000_000)")
    p.add_argument("--seed", type=int, default=None, help="Optional RNG seed for reproducibility")
    p.add_argument("--numpy", action="store_true", help="Use numpy arrays (if available) instead of Python list")
    return p.parse_args()

def python_sort(n, seed=None):
    if seed is not None:
        random.seed(seed)
    t0 = time.perf_counter()
    arr = 
    t_gen = time.perf_counter() - t0

    t_sort_start = time.perf_counter()
    arr.sort()  # in-place Timsort
    t_sort = time.perf_counter() - t_sort_start

    # verify sorted (fast generator-based check)
    t_check_start = time.perf_counter()
    ok = all(a <= b for a, b in zip(arr, arr[1:]))
    t_check = time.perf_counter() - t_check_start

    return {
        "method": "python_list",
        "n": n,
        "gen_time": t_gen,
        "sort_time": t_sort,
        "check_time": t_check,
        "sorted_ok": ok,
        "first10": arr[:10],
        "last10": arr[-10:],
    }

def numpy_sort(n, seed=None):
    try:
        import numpy as np
    except Exception as e:
        raise RuntimeError("NumPy not available; install it or run without --numpy") from e

    if seed is not None:
        # NumPy seed (keeps separate from random.seed)
        np.random.seed(seed)

    t0 = time.perf_counter()
    arr = np.random.random(size=n)
    t_gen = time.perf_counter() - t0

    t_sort_start = time.perf_counter()
    arr.sort()  # in-place numpy quicksort/mergesort (depends on dtype & algorithm)
    t_sort = time.perf_counter() - t_sort_start

    t_check_start = time.perf_counter()
    # vectorized check: all adjacent diffs >= 0
    ok = bool((arr[1:] >= arr[:-1]).all())
    t_check = time.perf_counter() - t_check_start

    return {
        "method": "numpy",
        "n": n,
        "gen_time": t_gen,
        "sort_time": t_sort,
        "check_time": t_check,
        "sorted_ok": ok,
        "first10": arr[:10].tolist(),
        "last10": arr[-10:].tolist(),
    }

def main():
    args = parse_args()
    n = args.n
    seed = args.seed
    use_numpy = args.numpy

    print(f"Sorting {n:,} random numbers (seed={seed!r}, use_numpy={use_numpy})")

    try:
        if use_numpy:
            result = numpy_sort(n, seed=seed)
        else:
            result = python_sort(n, seed=seed)
    except Exception as e:
        print("Error:", e, file=sys.stderr)
        sys.exit(1)

    print(f"Method: {result['method']}")
    print(f"Generation time: {result['gen_time']:.3f} s")
    print(f"Sort time:       {result['sort_time']:.3f} s")
    print(f"Check time:      {result['check_time']:.3f} s")
    print(f"Sorted OK:       {result['sorted_ok']}")
    print()
    print("First 10 elements:", result["first10"])
    print("Last  10 elements:", result["last10"])

if __name__ == "__main__":
    main()
```

Notes and tips:
- The default method uses Python's list and list.sort(), which is implemented with Timsort and is very efficient. 
Generating and sorting 1,000,000 floats typically completes in a few seconds on a modern desktop (times d

### reasoing 'minimal' 

o3 도 추론의 정도를 low, medium, high 옵션으로 조절 할 수 있었습니다. gpt5 에서는 추론에 최소한의 token을 사용하는 'minimal' 옵션을 사용할 수 있게 되었습니다.


In [107]:
query = "회사의 CEO가 가장 싫어하는 요일의 하루는 몇시간인가? 잘 생각하고 답변해줘"

minimal_respone = client.responses.create(
    model="gpt-5",
    input=query,
    reasoning={
        "effort": "minimal"
    }
)

high_respone = client.responses.create(
    model="gpt-5",
    input=query,
    reasoning={
        "effort": "high"
    }
)

In [108]:
pprint(minimal_respone.output_text)

정답: 8시간

해설:
회사의 CEO가 가장 싫어하는 요일은 보통 “월요일”이라고 말장난을 많이 합니다. 그런데 “월요일 하루는 몇 시간인가?”라고
물으면, 보통 하루는 24시간이지만 회사 관점에서 ‘근무하는 하루’는 8시간으로 보는 말장난 수수께끼입니다. 그래서 
정답은 8시간입니다.

In [109]:
pprint(minimal_respone.usage.model_dump())

{
    'input_tokens': 31,
    'input_tokens_details': {'cached_tokens': 0},
    'output_tokens': 99,
    'output_tokens_details': {'reasoning_tokens': 0},
    'total_tokens': 130
}

In [110]:
pprint(high_respone.output_text)

24시간입니다. 어떤 요일을 싫어하든 하루는 모두 24시간이에요.

In [111]:
pprint(high_respone.usage.model_dump())

{
    'input_tokens': 31,
    'input_tokens_details': {'cached_tokens': 0},
    'output_tokens': 2844,
    'output_tokens_details': {'reasoning_tokens': 2816},
    'total_tokens': 2875
}

### Freeform

tools type 을 'custom' 이라고 해주면, tool calling 을 응용해서 구조화된 출력없이 바로 결과물을 받아 볼 수 있다는 것이다.

In [145]:
query = "Please use the code_exec tool to calculate the area of a circle with radius equal to the number of 'r's in strawberry"
response = client.responses.create(
    model="gpt-5-mini",
    input=query,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "code_exec",
            "description": "Executes arbitrary python code",
        }
    ]
)

In [146]:
pprint(response.output[1].input)

# Calculate the area of a circle where the radius is the number of 'r' characters in "strawberry"
import math
word = "strawberry"
r = word.count('r')
area = math.pi * r**2
# Return results as a tuple: radius, exact symbolic, numeric area
(r, f"{r}^2 * pi", area)

전에는 tool 을 명시해서 받아왔어야 했다.

In [127]:
query = "Please use the code_exec tool to calculate the area of a circle with radius equal to the number of 'r's in strawberry"
no_freeform_response = client.responses.create(
    model="gpt-4o-mini",
    input=query,
    tools = [
        {
            "type": "function",
            "name": "code_exec",
            "description": "Executes arbitrary python code",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Python code to execute",
                    },
                },
                "required": ["code"],
            },
        },
    ]
)

In [149]:
import json

python_code = json.loads(no_freeform_response.output[0].arguments)['code']
python_code

"import math\nradius = 'strawberry'.count('r')\narea = math.pi * (radius ** 2)\narea"

### Context-Free Grammer(CFG)

문법 규칙이 주변 맥락에 의존하지 않는다는 의미로 Context free 라는 의미로 사용된다. 프로그래밍 언어 파싱, 수식 파싱 등에 주로 사용된다고 합니다.
룰 기반으로 모델의 출력을 조절 할 수 있어서, 신뢰도가 높은 합성 데이터 등 특정 케이스에는 아주 유용하게 사용 할 수 있을 것 같습니다. 아래 예제는, 아주 적합한 사용 예시는 아니지만,
보다 쉽게 받아 들일 수 있도록 예시를 들어봤습니다. 

In [151]:
import textwrap

In [154]:
coupang_data_grammar = textwrap.dedent(r"""
start: shipping_with_valid_id | shipping_with_invalid_id

shipping_with_valid_id: "주문번호 " ORDER_ID12 " 의 현재 상태는 " STATUS " 입니다."
shipping_with_invalid_id: "주문번호 " (INVALID_TOO_SHORT | INVALID_TOO_LONG | INVALID_NON_DIGIT) " 는 유효하지 않은 주문번호입니다. 숫자 12자리를 입력해주세요."
STATUS: "결제완료" | "배송준비중" | "배송중" | "배송완료"


ORDER_ID12: /[0-9]{12}/
INVALID_TOO_SHORT: /[0-9]{1,11}/
INVALID_TOO_LONG: /[0-9]{13,}/
INVALID_NON_DIGIT: /\S*\D\S*/
""")

custcenter_prompt = "`customer_service_grammar` 를 사용해서 유효하지 않은 주문번호를 사용한 응답을 만들어라"

response_coup = client.responses.create(
    model="gpt-5-mini",
    input=custcenter_prompt,
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "customer_service_grammar",
            "description": "고객센터 챗봇의 응답 데이터를 만드는 문법입니다.",
            "format": {
                "type": "grammar",
                "syntax": "lark",
                "definition": coupang_data_grammar
            }
        },
        
    ],
    parallel_tool_calls=False
)

print(response_coup.output[1].input)

주문번호 12345 는유효하지않은주문번호입니다.숫자 는 유효하지 않은 주문번호입니다. 숫자 12자리를 입력해주세요.
